# JEWEL events and FastJet in Python

Activate the Conda environment or virtual environment you’ll use as the notebook kernel. With cppyy available, install JEWEL and the FastJet dependencies from PyPI:

```bash
pip install 'hepyy[hepbundle,jewel-cpp]'
```

If the environment needs cppyy from pip, use `pip install 'hepyy[cppyy,hepbundle,jewel-cpp]'`. Launch Jupyter from the configured environment and select its kernel.

For a source build, follow the [hepyy recipe fallback](../README.md#hepyy-recipe-fallback) or the [manual CMake fallback](../README.md#manual-cmake-fallback).

In [ ]:
import hepyy

hepyy.load(["jewel_cpp", "fastjet"])

import fastjet
import jewel_cpp

config = jewel_cpp.EventGeneratorConfig()
config.input.seed = 2601301
config.input.initial_state_radiation = True
generator = jewel_cpp.EventGenerator(config)
jet_definition = fastjet.JetDefinition(fastjet.antikt_algorithm, 0.4)
particle_vector = fastjet.gbl.std.vector[fastjet.PseudoJet]

Cluster final particles over full phase space, excluding neutrinos. Select jets with pT ≥ 20 GeV and |η| < 2.

In [ ]:
for _ in range(10):
    event = generator.next()
    if not event.accepted:
        print(f"Attempt {event.attempt} failed in {event.stage}: {event.error}")
        continue

    particles = particle_vector()
    for particle in event.particles:
        if abs(particle.pdg) in (12, 14, 16):
            continue
        p = particle.momentum
        particles.push_back(fastjet.PseudoJet(p.px, p.py, p.pz, p.energy))

    sequence = fastjet.ClusterSequence(particles, jet_definition)
    jets = fastjet.sorted_by_pt(sequence.inclusive_jets(20.0))
    pts = [round(jet.pt(), 2) for jet in jets if abs(jet.eta()) < 2.0]
    print(f"Event {event.attempt}, weight {event.weight:.6g}, jet pT [GeV] {pts}")

In [ ]:
stats = generator.hard_statistics()
print(f"Hard input: {stats.returned}/{stats.attempts} events, sigma = {stats.sigma_mb:.6g} mb")